# 02 - Nettoyage et validation des données

## Objectif

Cette analyse vérifie la qualité du dataset avant les analyses approfondies.

Nous allons notamment :

- rechercher les doublons ;
- analyser les valeurs manquantes ;
- vérifier les formats ;
- détecter les anomalies ;
- valider le dataset nettoyé.


## 1. Importation des bibliothèques

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


## 2. Chargement des données

In [2]:
df = pd.read_csv("../data/processed/netflix_titles_clean.csv")

print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")


Dimensions : 8807 lignes × 12 colonnes


## 3. Vérification des doublons

In [3]:
duplicates = df.duplicated().sum()

print(f"Nombre de doublons : {duplicates}")


Nombre de doublons : 0


In [4]:
df[df.duplicated(keep=False)].head(20)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description


## 4. Vérification des valeurs manquantes

In [5]:
missing = df.isna().sum()

missing_df = (
    missing[missing > 0]
    .sort_values(ascending=False)
    .to_frame("Nombre de valeurs manquantes")
)

missing_df


,Nombre de valeurs manquantes
date_added,10
rating,4
duration,3


In [6]:
missing_percentage = (
    df.isna().mean() * 100
).round(2)

missing_percentage[missing_percentage > 0].sort_values(ascending=False)


date_added    0.11
rating        0.05
duration      0.03
dtype: float64

## 5. Vérification des types de données

In [7]:
df.dtypes


show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

## 6. Vérification des valeurs uniques

In [8]:
for column in ["type", "rating"]:
    print(f"\n=== {column.upper()} ===")
    print(df[column].value_counts(dropna=False).to_string())



=== TYPE ===
type
Movie      6131
TV Show    2676

=== RATING ===
rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NaN            4
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1


## 7. Détection des anomalies dans la classification

Lors de l'exploration, certaines valeurs présentes dans `rating` semblaient correspondre à des durées de films.

Nous allons vérifier ces valeurs.


In [9]:
suspicious_ratings = df[
    df["rating"].astype(str).str.match(r"^\d+ min$", na=False)
]

suspicious_ratings[["show_id", "title", "type", "rating", "duration"]]


,show_id,title,type,rating,duration
5541,s5542,Louis C.K. 2017,Movie,74 min,NaN
5794,s5795,Louis C.K.: Hilarious,Movie,84 min,NaN
5813,s5814,Louis C.K.: Live at the Comedy Store,Movie,66 min,NaN


## 8. Vérification de la colonne `duration`

In [10]:
duration_missing = df["duration"].isna().sum()

print(f"Durées manquantes : {duration_missing}")
print("\nExemples de durées :")
print(df["duration"].value_counts().head(20).to_string())


Durées manquantes : 3

Exemples de durées :
duration
1 Season     1793
2 Seasons     425
3 Seasons     199
90 min        152
94 min        146
97 min        146
93 min        146
91 min        144
95 min        137
96 min        130
92 min        129
102 min       122
98 min        120
99 min        118
88 min        116
101 min       116
103 min       114
106 min       111
100 min       108
89 min        106


## 9. Vérification des dates

In [11]:
df["date_added"] = pd.to_datetime(
    df["date_added"],
    errors="coerce"
)

print("Type de date après conversion :", df["date_added"].dtype)
print("Dates manquantes :", df["date_added"].isna().sum())


Type de date après conversion : datetime64[us]
Dates manquantes : 10


## 10. Validation finale

In [12]:
print("=== VALIDATION DU DATASET ===")
print(f"Lignes : {df.shape[0]}")
print(f"Colonnes : {df.shape[1]}")
print(f"Doublons : {df.duplicated().sum()}")
print(f"Valeurs manquantes totales : {df.isna().sum().sum()}")


=== VALIDATION DU DATASET ===
Lignes : 8807
Colonnes : 12
Doublons : 0
Valeurs manquantes totales : 17


## 11. Conclusion

Le dataset a été inspecté afin d'identifier les principaux problèmes de qualité.

Les anomalies détectées seront prises en compte avant les analyses approfondies. Le dataset nettoyé constitue désormais la base de travail pour les prochains notebooks.

Les analyses spécialisées seront séparées afin de conserver une structure claire et professionnelle.
